# EDA - Segmentacion de Miocardio (ACDC + York)

Exploracion de datos del dataset preprocesado (train / val / test), usando el pipeline
definido en `cardiac_dataset.py`, y resultados del modelo de la etapa 1.

Si en `cardiac_data/` faltan algunos splits (por ejemplo, solo esta `val`), el notebook
trabaja con los que haya y normaliza con la media y el std guardados en el checkpoint,
que son los mismos que se calcularon en train.

Contenido:
1. Carga de datos y resumen general (pandas)
2. Balance de clases por split
3. Visualizacion de muestras (imagen, mascara 3 clases, mascara binaria)
4. Distribucion del area de miocardio por imagen
5. Curvas de entrenamiento
6. Resultado final en test
7. Predicciones del modelo

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, ".")  # para poder importar los modulos del proyecto
from cardiac_dataset import SPLITS, load_splits, class_balance
from model import load_checkpoint

DATA_DIR = Path("cardiac_data")  # <-- cambia esto por tu ruta si es distinta
OUT = Path("outputs")

presentes = [s for s in SPLITS if (DATA_DIR / f"{s}_imgs.npy").exists()]
if "train" in presentes:
    data = load_splits(str(DATA_DIR), splits=presentes)
else:
    # sin train no se pueden recalcular los stats: se usan los del checkpoint
    _, ckpt = load_checkpoint(OUT / "best_model.pt")
    data = load_splits(str(DATA_DIR), stats={"mean": ckpt["mean"], "std": ckpt["std"]},
                       splits=presentes)

# split para las secciones 3 y 4: train si esta, si no el primero disponible
SPLIT_EDA = "train" if "train" in presentes else presentes[0]
print("Splits cargados:", presentes)
print("Stats de normalizacion (calculadas en train):", data["_stats"])

## 1. Resumen general con pandas

In [ ]:
rows = []
for split in presentes:
    sd = data[split]
    bg_frac, fg_frac = class_balance(sd.masks)
    rows.append({
        "split": split,
        "n_imagenes": sd.images.shape[0],
        "alto": sd.images.shape[1],
        "ancho": sd.images.shape[2],
        "pct_fondo": round(bg_frac * 100, 2),
        "pct_miocardio": round(fg_frac * 100, 2),
    })

df_resumen = pd.DataFrame(rows).set_index("split")
df_resumen

## 2. Balance de clases por split

El dataset esta fuertemente desbalanceado: el miocardio ocupa solo ~1-1.5% de los
pixeles. Esto es importante para elegir la funcion de perdida del modelo (Dice/IoU
loss o weighted cross-entropy suelen funcionar mejor que cross-entropy plano aqui).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df_resumen[["pct_fondo", "pct_miocardio"]].plot(
    kind="bar", stacked=True, ax=ax, color=["#4C72B0", "#DD8452"]
)
ax.set_ylabel("% de pixeles")
ax.set_title("Balance de clases por split")
ax.legend(["Fondo", "Miocardio"])
plt.tight_layout()
plt.show()


## 3. Muestras: imagen original, mascara 3 clases, mascara binaria

In [ ]:
raw_annot = np.load(DATA_DIR / f"{SPLIT_EDA}_annot.npy", mmap_mode="r")
raw_imgs = np.load(DATA_DIR / f"{SPLIT_EDA}_imgs.npy", mmap_mode="r")

# elegimos algunas imagenes con harto miocardio visible, para verificar la mascara
fg_counts = (raw_annot[:] == 1).reshape(len(raw_annot), -1).sum(axis=1)
idxs = np.argsort(-fg_counts)[:4]

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for col, idx in enumerate(idxs):
    img = raw_imgs[idx]
    ann = raw_annot[idx]
    bin_mask = (ann == 1).astype(np.uint8)

    axes[0, col].imshow(img, cmap="gray")
    axes[0, col].set_title(f"{SPLIT_EDA} idx {idx}: original", fontsize=9)
    axes[0, col].axis("off")

    axes[1, col].imshow(img, cmap="gray")
    axes[1, col].imshow(np.ma.masked_where(ann == 0, ann), cmap="jet", alpha=0.6, vmin=0, vmax=2)
    axes[1, col].set_title("mascara 3 clases", fontsize=9)
    axes[1, col].axis("off")

    axes[2, col].imshow(img, cmap="gray")
    axes[2, col].imshow(np.ma.masked_where(bin_mask == 0, bin_mask), cmap="autumn", alpha=0.6)
    axes[2, col].set_title("mascara binaria (miocardio)", fontsize=9)
    axes[2, col].axis("off")

plt.tight_layout()
plt.show()

## 4. Distribucion del area de miocardio por imagen

Cuantos pixeles de miocardio tiene cada imagen (en train si esta disponible). Sirve para
detectar imagenes atipicas (por ejemplo, cortes donde el miocardio casi no aparece, o
imagenes sin miocardio en absoluto que podrian ser puramente de fondo).

In [ ]:
masks = data[SPLIT_EDA].masks
areas = masks.reshape(len(masks), -1).sum(axis=1)
df_areas = pd.Series(areas, name="pixeles_miocardio")

print(df_areas.describe())

fig, ax = plt.subplots(figsize=(7, 4))
df_areas.plot(kind="hist", bins=40, ax=ax, color="#55A868")
ax.set_xlabel("Pixeles de miocardio por imagen")
ax.set_title(f"Distribucion del area de miocardio ({SPLIT_EDA})")
plt.tight_layout()
plt.show()

n_sin_miocardio = (df_areas == 0).sum()
print(f"Imagenes sin miocardio en {SPLIT_EDA}: {n_sin_miocardio} ({n_sin_miocardio/len(df_areas):.2%})")

## Proximos pasos

- Definir arquitectura encoder-decoder (U-Net) para segmentacion binaria.
- Elegir loss apropiada para el desbalance de clases (Dice loss / IoU loss / weighted BCE).
- Entrenar y evaluar con la metrica IoU promedio sobre val/test.

## 5. Curvas de entrenamiento

Carga el historial de metricas (`outputs/history.json`) generado por `train.py`
y grafica la evolucion de loss e IoU en train/val por epoca.

In [ ]:
with open(OUT / "history.json") as f:
    history = json.load(f)

df_hist = pd.DataFrame(history).set_index("epoch")
df_hist

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_hist[["train_loss", "val_loss"]].plot(ax=axes[0], marker="o")
axes[0].set_title("Loss por epoca")
axes[0].set_ylabel("BCE + Dice loss")
axes[0].grid(alpha=0.3)

df_hist[["train_iou", "val_iou"]].plot(ax=axes[1], marker="o", color=["#4C72B0", "#DD8452"])
axes[1].set_title("IoU por epoca")
axes[1].set_ylabel("IoU (Jaccard)")
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mejor val_iou: {df_hist['val_iou'].max():.4f} en la epoca {df_hist['val_iou'].idxmax()}")


## 6. Resultado final en test

In [ ]:
with open(OUT / "test_results.json") as f:
    test_results = json.load(f)

pd.Series(test_results)

## 7. Predicciones del modelo

Genera la figura con `visualize_predictions.py` sobre test (o sobre val si test no esta en
disco) y la muestra aqui. Verde = acierto, rojo = falso positivo, azul = falso negativo.

In [ ]:
import subprocess

SPLIT_VIS = "test" if "test" in presentes else "val"
result = subprocess.run(
    [sys.executable, "visualize_predictions.py", str(DATA_DIR),
     "--checkpoint", str(OUT / "best_model.pt"), "--n", "6",
     "--split", SPLIT_VIS, "--out-dir", str(OUT)],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
result.check_returncode()

from IPython.display import Image
Image(str(OUT / f"predicciones_{SPLIT_VIS}_aleatorio.png"))